# EDA-02 · Return Analysis
Return rate %, top return reasons, category return heatmap, net revenue after returns.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 10,
})

CAT_COLORS = ['#4C72B0','#DD8452','#55A868','#C44E52']

# ── Load ─────────────────────────────────────────────────────────────────────
ret    = pd.read_csv('returns.csv',    parse_dates=['return_date'])
orders = pd.read_csv('orders.csv',     parse_dates=['order_date'])
items  = pd.read_csv('order_items.csv', low_memory=False)
prods  = pd.read_csv('products.csv')
pay    = pd.read_csv('payments.csv')
sales  = pd.read_csv('sales.csv',      parse_dates=['Date'])

# ── Master item table ─────────────────────────────────────────────────────────
items_full = (
    items
    .merge(orders[['order_id','order_date','order_status']], on='order_id', how='left')
    .merge(prods[['product_id','category','segment','price','cogs']], on='product_id', how='left')
)
items_full['revenue_line'] = items_full['quantity'] * items_full['unit_price']
items_full['cogs_line']    = items_full['quantity'] * items_full['cogs']
items_full['year']         = items_full['order_date'].dt.year

# ── Return master ─────────────────────────────────────────────────────────────
ret_full = (
    ret
    .merge(orders[['order_id','order_date']], on='order_id', how='left')
    .merge(prods[['product_id','category','segment']], on='product_id', how='left')
    .merge(pay[['order_id','payment_value']], on='order_id', how='left')
)
ret_full['year'] = ret_full['order_date'].dt.year

print(f'orders  : {len(orders):,}')
print(f'returns : {len(ret_full):,}  (unique orders: {ret_full["order_id"].nunique():,})')
print(f'items   : {len(items_full):,}')


## 1. Overall Return Rate

In [ ]:
total_orders   = len(orders)
returned_orders = orders[orders['order_status']=='returned']['order_id'].nunique()
del_ret_orders  = orders[orders['order_status'].isin(['delivered','returned'])]['order_id'].nunique()

total_qty   = items_full['quantity'].sum()
returned_qty = ret_full['return_quantity'].sum()

gross_rev  = items_full['revenue_line'].sum()
total_refund = ret_full['refund_amount'].sum()

print('=== Overall Return Metrics ===')
print(f'Return rate (order-level, returned / all)            : {returned_orders/total_orders*100:.2f}%')
print(f'Return rate (order-level, returned / delivered+ret)  : {returned_orders/del_ret_orders*100:.2f}%')
print(f'Return rate (item quantity, qty_returned / qty_sold) : {returned_qty/total_qty*100:.2f}%')
print(f'Return rate (value, refund / gross revenue)          : {total_refund/gross_rev*100:.2f}%')
print()
print(f'Total refund amount : {total_refund:,.0f} VND')
print(f'Avg refund/return   : {total_refund/len(ret_full):,.0f} VND')

## 2. Return Rate Trend by Year

In [ ]:
yearly_orders  = orders.groupby(orders['order_date'].dt.year)['order_id'].count().rename('total')
yearly_returned = orders[orders['order_status']=='returned'].groupby(
    orders['order_date'].dt.year)['order_id'].count().rename('returned')
yearly_refund  = ret_full.groupby('year')['refund_amount'].sum()
yearly_rev     = sales.groupby(sales['Date'].dt.year)['Revenue'].sum()

trend = pd.concat([yearly_orders, yearly_returned], axis=1).fillna(0)
trend['return_rate_%'] = (trend['returned'] / trend['total'] * 100).round(2)
trend['refund_VND']    = yearly_refund
trend['gross_rev']     = yearly_rev
trend['refund_rate_%'] = (trend['refund_VND'] / trend['gross_rev'] * 100).round(2)
print(trend.to_string())

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].bar(trend.index, trend['return_rate_%'], color='#C44E52', alpha=0.85)
axes[0].set_xlabel('Year'); axes[0].set_ylabel('Return rate %')
axes[0].set_title('Order return rate by year')
for i, (yr, v) in enumerate(trend['return_rate_%'].items()):
    axes[0].text(yr, v + 0.05, f'{v:.1f}%', ha='center', fontsize=8)

axes[1].bar(trend.index, trend['refund_rate_%'], color='#DD8452', alpha=0.85)
axes[1].set_xlabel('Year'); axes[1].set_ylabel('Refund / revenue %')
axes[1].set_title('Refund value rate by year (refund / gross revenue)')
for i, (yr, v) in enumerate(trend['refund_rate_%'].items()):
    if not pd.isna(v):
        axes[1].text(yr, v + 0.01, f'{v:.1f}%', ha='center', fontsize=8)
plt.tight_layout(); plt.show()

## 3. Top Return Reasons

In [ ]:
reason_stats = (
    ret_full.groupby('return_reason')
    .agg(count=('return_id','count'),
         total_qty=('return_quantity','sum'),
         total_refund=('refund_amount','sum'))
    .sort_values('count', ascending=False)
    .reset_index()
)
reason_stats['count_%']  = reason_stats['count']  / len(ret_full) * 100
reason_stats['refund_%'] = reason_stats['total_refund'] / ret_full['refund_amount'].sum() * 100
print(reason_stats.round(2).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

colors_r = ['#C44E52','#DD8452','#4C72B0','#55A868','#9467BD']
axes[0].barh(reason_stats['return_reason'], reason_stats['count'], color=colors_r)
axes[0].set_xlabel('Number of returns')
axes[0].set_title('Return count by reason')
for bar, v, p in zip(axes[0].patches, reason_stats['count'], reason_stats['count_%']):
    axes[0].text(bar.get_width() + 50, bar.get_y() + bar.get_height()/2,
                 f'{v:,} ({p:.1f}%)', va='center', fontsize=8)

axes[1].barh(reason_stats['return_reason'], reason_stats['total_refund']/1e6, color=colors_r)
axes[1].set_xlabel('Refund amount (M VND)')
axes[1].set_title('Refund value by reason')
for bar, v, p in zip(axes[1].patches, reason_stats['total_refund']/1e6, reason_stats['refund_%']):
    axes[1].text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                 f'{v:.0f}M ({p:.1f}%)', va='center', fontsize=8)
plt.tight_layout(); plt.show()

## 4. Return Reason Trend by Year

In [ ]:
reason_yr = (
    ret_full.groupby(['year','return_reason'])['return_id']
    .count().unstack(fill_value=0)
)
reason_yr_pct = reason_yr.div(reason_yr.sum(axis=1), axis=0) * 100
print('Return reason share by year (%):')
print(reason_yr_pct.round(1).to_string())

fig, ax = plt.subplots(figsize=(11, 4))
reason_yr_pct.plot(kind='bar', stacked=True, ax=ax,
                   color=colors_r, edgecolor='white', linewidth=0.4)
ax.set_xlabel('Year'); ax.set_ylabel('Share %')
ax.set_title('Return reason composition by year (stacked %)')
ax.legend(title='Reason', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
ax.tick_params(axis='x', rotation=0)
plt.tight_layout(); plt.show()

## 5. Return Rate by Category

In [ ]:
# Qty sold vs qty returned per category
qty_sold = items_full.groupby('category')['quantity'].sum().rename('qty_sold')
qty_ret  = ret_full.groupby('category')['return_quantity'].sum().rename('qty_returned')
cat_stats = pd.concat([qty_sold, qty_ret], axis=1).fillna(0)
cat_stats['return_rate_%'] = (cat_stats['qty_returned'] / cat_stats['qty_sold'] * 100).round(2)

rev_by_cat    = items_full.groupby('category')['revenue_line'].sum().rename('gross_rev')
refund_by_cat = ret_full.groupby('category')['refund_amount'].sum().rename('refund')
cat_stats = cat_stats.join(rev_by_cat).join(refund_by_cat)
cat_stats['refund_rate_%'] = (cat_stats['refund'] / cat_stats['gross_rev'] * 100).round(2)

print(cat_stats.round(2).to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
cats = cat_stats.index.tolist()

axes[0].bar(cats, cat_stats['return_rate_%'], color=CAT_COLORS[:len(cats)])
axes[0].set_ylabel('Return rate (qty) %')
axes[0].set_title('Return rate by category (quantity-based)')
for i, v in enumerate(cat_stats['return_rate_%']):
    axes[0].text(i, v + 0.05, f'{v:.2f}%', ha='center', fontsize=9)

axes[1].bar(cats, cat_stats['refund_rate_%'], color=CAT_COLORS[:len(cats)])
axes[1].set_ylabel('Refund / gross revenue %')
axes[1].set_title('Refund rate by category (value-based)')
for i, v in enumerate(cat_stats['refund_rate_%']):
    axes[1].text(i, v + 0.05, f'{v:.2f}%', ha='center', fontsize=9)
plt.tight_layout(); plt.show()

## 6. Category × Return Reason Heatmap

In [ ]:
heat = (
    ret_full.groupby(['category','return_reason'])['return_id']
    .count().unstack(fill_value=0)
)
# Normalize to return rate % within each category (per qty sold)
qty_sold_cat = items_full.groupby('category')['quantity'].sum()
heat_pct = heat.div(qty_sold_cat, axis=0) * 100

print('Return reason rate % (returns / qty sold):')
print(heat_pct.round(3).to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# absolute count heatmap
im1 = axes[0].imshow(heat.values, cmap='Reds', aspect='auto')
axes[0].set_xticks(range(len(heat.columns))); axes[0].set_xticklabels(heat.columns, rotation=25, ha='right', fontsize=8)
axes[0].set_yticks(range(len(heat.index)));   axes[0].set_yticklabels(heat.index)
axes[0].set_title('Return count: category × reason')
plt.colorbar(im1, ax=axes[0], shrink=0.8)
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        axes[0].text(j, i, f'{heat.values[i,j]:,}', ha='center', va='center',
                     fontsize=8, color='white' if heat.values[i,j] > heat.values.max()*0.6 else 'black')

# rate % heatmap
im2 = axes[1].imshow(heat_pct.values, cmap='YlOrRd', aspect='auto')
axes[1].set_xticks(range(len(heat_pct.columns))); axes[1].set_xticklabels(heat_pct.columns, rotation=25, ha='right', fontsize=8)
axes[1].set_yticks(range(len(heat_pct.index)));   axes[1].set_yticklabels(heat_pct.index)
axes[1].set_title('Return rate % (/ qty sold): category × reason')
plt.colorbar(im2, ax=axes[1], shrink=0.8)
for i in range(heat_pct.shape[0]):
    for j in range(heat_pct.shape[1]):
        axes[1].text(j, i, f'{heat_pct.values[i,j]:.2f}%', ha='center', va='center',
                     fontsize=8, color='white' if heat_pct.values[i,j] > heat_pct.values.max()*0.6 else 'black')
plt.tight_layout(); plt.show()

## 7. Category × Segment Return Rate Heatmap

In [ ]:
qty_sold_seg = items_full.groupby(['category','segment'])['quantity'].sum().rename('qty_sold')
qty_ret_seg  = ret_full.groupby(['category','segment'])['return_quantity'].sum().rename('qty_ret')
seg_heat = pd.concat([qty_sold_seg, qty_ret_seg], axis=1).fillna(0)
seg_heat['rate_%'] = (seg_heat['qty_ret'] / seg_heat['qty_sold'] * 100).round(2)

pivot = seg_heat['rate_%'].unstack(level='segment').fillna(0)
print('Return rate % by category × segment:')
print(pivot.round(2).to_string())

fig, ax = plt.subplots(figsize=(11, 4))
im = ax.imshow(pivot.values, cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns, rotation=30, ha='right', fontsize=9)
ax.set_yticks(range(len(pivot.index)));   ax.set_yticklabels(pivot.index, fontsize=9)
ax.set_title('Return rate % (qty): category × segment')
plt.colorbar(im, ax=ax, shrink=0.7, label='Return rate %')
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        v = pivot.values[i, j]
        if v > 0:
            ax.text(j, i, f'{v:.1f}%', ha='center', va='center',
                    fontsize=8, color='white' if v > pivot.values.max()*0.65 else 'black')
plt.tight_layout(); plt.show()

## 8. Net Revenue After Returns

In [ ]:
# Gross revenue = sales.csv (all orders)
# Refund grouped by order_date (khi nao order bi return, tinh refund vao ngay dat hang)
refund_by_order_date = (
    ret_full.merge(orders[['order_id','order_date']], on='order_id', how='left')
    .groupby(orders['order_date'].dt.to_period('M').rename('month'))['refund_amount']
    .sum()
    .reset_index()
)
refund_by_order_date.columns = ['month','refund']
refund_by_order_date['month_ts'] = refund_by_order_date['month'].dt.to_timestamp()

# Monthly gross revenue from sales.csv
sales['month'] = sales['Date'].dt.to_period('M')
monthly_rev = sales.groupby('month')[['Revenue','COGS']].sum().reset_index()
monthly_rev['month_ts'] = monthly_rev['month'].dt.to_timestamp()

df_net = monthly_rev.merge(refund_by_order_date[['month','refund']], on='month', how='left').fillna(0)
df_net['net_revenue'] = df_net['Revenue'] - df_net['refund']
df_net['refund_rate_%'] = df_net['refund'] / df_net['Revenue'] * 100

print('Monthly net revenue sample (last 12 months):')
print(df_net[['month','Revenue','refund','net_revenue','refund_rate_%']].tail(12).round(0).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

axes[0].fill_between(df_net['month_ts'], df_net['Revenue']/1e6,
                     alpha=0.3, color='#4C72B0', label='Gross revenue')
axes[0].fill_between(df_net['month_ts'], df_net['net_revenue']/1e6,
                     alpha=0.6, color='#55A868', label='Net revenue (after refund)')
axes[0].set_ylabel('Revenue (M VND)')
axes[0].set_title('Monthly Gross vs Net Revenue')
axes[0].legend(loc='upper left', fontsize=9)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:.0f}M'))

axes[1].fill_between(df_net['month_ts'], df_net['refund_rate_%'],
                     alpha=0.7, color='#C44E52', label='Refund / Revenue %')
axes[1].set_ylabel('Refund rate %')
axes[1].set_title('Monthly refund rate (refund / gross revenue)')
axes[1].legend(loc='upper left', fontsize=9)
plt.tight_layout(); plt.show()

## 9. Net Revenue by Category (Annual)

In [ ]:
gross_cat_yr  = items_full.groupby(['year','category'])['revenue_line'].sum().unstack(fill_value=0)
refund_cat_yr = ret_full.groupby(['year','category'])['refund_amount'].sum().unstack(fill_value=0)
net_cat_yr    = gross_cat_yr.subtract(refund_cat_yr, fill_value=0)

print('Annual gross revenue by category (M VND):')
print((gross_cat_yr/1e6).round(1).to_string())
print()
print('Annual refund by category (M VND):')
print((refund_cat_yr/1e6).round(1).to_string())
print()
print('Annual net revenue by category (M VND):')
print((net_cat_yr/1e6).round(1).to_string())

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

(gross_cat_yr/1e6).plot(kind='bar', ax=axes[0], color=CAT_COLORS[:gross_cat_yr.shape[1]],
                        edgecolor='white', linewidth=0.4)
axes[0].set_title('Annual gross revenue by category (M VND)')
axes[0].set_xlabel('Year'); axes[0].set_ylabel('M VND')
axes[0].legend(title='Category', fontsize=8)
axes[0].tick_params(axis='x', rotation=0)

(net_cat_yr/1e6).plot(kind='bar', ax=axes[1], color=CAT_COLORS[:net_cat_yr.shape[1]],
                      edgecolor='white', linewidth=0.4)
axes[1].set_title('Annual net revenue by category (M VND)')
axes[1].set_xlabel('Year'); axes[1].set_ylabel('M VND')
axes[1].legend(title='Category', fontsize=8)
axes[1].tick_params(axis='x', rotation=0)
plt.tight_layout(); plt.show()

## Summary

In [ ]:
total_gross  = items_full['revenue_line'].sum()
total_refund = ret_full['refund_amount'].sum()
total_net    = total_gross - total_refund

print('=== Return Analysis Summary ===')
print(f'Total gross revenue  : {total_gross:>18,.0f} VND')
print(f'Total refunds        : {total_refund:>18,.0f} VND  ({total_refund/total_gross*100:.2f}%)')
print(f'Total net revenue    : {total_net:>18,.0f} VND')
print()
print(f'Order return rate    : {returned_orders/total_orders*100:.2f}%  ({returned_orders:,} / {total_orders:,} orders)')
print(f'Qty return rate      : {returned_qty/total_qty*100:.2f}%  ({int(returned_qty):,} / {int(total_qty):,} units)')
print()
print('Top return reason    :', reason_stats.iloc[0]["return_reason"], f'({reason_stats.iloc[0]["count_%"]:.1f}%)')
print()
print('Return rate by category:')
for cat, row in cat_stats.iterrows():
    print(f'  {cat:<12}  qty_rate={row["return_rate_%"]:.2f}%  refund_rate={row["refund_rate_%"]:.2f}%')